<a href="https://colab.research.google.com/github/lyogavin/Anima/blob/main/air_llm/tests/test_notebooks/test_compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install airllm

In [1]:
!pip install -U airllm

In [2]:
!pip show airllm

Name: airllm
Version: 2.11.0
Summary: AirLLM allows single 4GB GPU card to run 70B large language models without quantization, distillation or pruning. 8GB vmem to run 405B Llama3.1.
Home-page: https://github.com/lyogavin/airllm
Author: Gavin Li
Author-email: gavinli@animaai.cloud
License: 
Location: /home/alonso/airllm/.venv/lib/python3.13/site-packages
Editable project location: /home/alonso/airllm/air_llm
Requires: accelerate, huggingface-hub, optimum, safetensors, scipy, torch, tqdm, transformers
Required-by: 


In [3]:
# This copy command is useful in some Colab workflows but not needed locally.
import os
from pathlib import Path

local_py_files = list(Path('.').glob('*.py'))
if local_py_files:
    print(f"Found local .py files: {[p.name for p in local_py_files]}")
else:
    print("No local .py files to copy; continuing.")

No local .py files to copy; continuing.


In [4]:
!pip install -U bitsandbytes
!python -c "import bitsandbytes as bnb; print('bitsandbytes importable:', bnb.__version__)"

bitsandbytes importable: 0.49.2


In [5]:
import torch
from airllm import AutoModel

MAX_LENGTH = 128

# Guard this notebook test so it fails only on model logic, not optional runtime availability.
try:
    import bitsandbytes as bnb  # noqa: F401
    bnb_available = True
except Exception as ex:
    bnb_available = False
    print(f"Skipping compression test because bitsandbytes is not usable: {ex}")

if bnb_available:
    model = AutoModel.from_pretrained("garage-bAInd/Platypus2-7B", compression='4bit')

    input_text = [
        'I like',
    ]

    input_tokens = model.tokenizer(
        input_text,
        return_tensors="pt",
        return_attention_mask=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = input_tokens['input_ids']
    if torch.cuda.is_available():
        input_ids = input_ids.cuda()

    generation_output = model.generate(
        input_ids,
        max_new_tokens=3,
        use_cache=True,
        return_dict_in_generate=True,
    )

    print(model.tokenizer.decode(generation_output.sequences[0]))

/home/alonso/airllm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>>> bitsandbytes installed
>>>> cache_utils installed


Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 106799.41it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.22.': True, 'model.layers.23.': True, 'model.layers.24.': True, 'model.layers.25.': True, 'model.layers.26.': True, 'model.layers.27.': True, 'model.layers.28.': True, 'model.layers.29.': True, 'model.layers.30.': True, 'model.layers.31.': True, 'model.norm.': True, 'lm_head.': True}
saved layers already found in /home/alonso/.cache/huggingface/hub/models--garage-bAInd--Platypus

optimum.bettertransformer is unavailable; falling back to standard model initialization
either BetterTransformer or attn_implementation='sdpa' is available, creating model directly
not support prefetching for compression for now. loading with no prepetching mode.


TypeError: 'DynamicCache' object is not subscriptable